# Airline EDA Case Study — Detailed Instructor Answer Key

This answer key is based on the uploaded **final learner notebook** and the uploaded **SQL-based notebook and its executed outputs**. It translates the original SQL analyses into the corresponding pandas/CSV interpretation.

> **Important:** The original executed learner notebook had an analytical issue: it replaced missing boarding observations with zero before calculating occupancy. The correct interpretation is that a missing boarding record is not automatically zero passengers. Therefore, use the corrected `occupancy_data` population in the final learner notebook for occupancy and route-level analysis.

## How learners should write an inference

**Finding → Meaning → Possible Action**

A good inference reports what the data shows, explains why it matters for the business, and suggests a reasonable investigation or test. It should not claim causality when the analysis only shows association.

# 1. Data Understanding — Answers

### Learner checkpoint

**Which table contains aircraft capacity information?**

`seats.csv`. Each row represents one seat for an aircraft type. Counting rows by `aircraft_code` gives the available seat capacity.

**Which table tells us how many passengers boarded a flight?**

`boarding_passes.csv`. Each boarding record represents a passenger boarding assignment for a flight. Grouping by `flight_id` gives the boarded-passenger count used for occupancy analysis.

**Which table contains ticket revenue?**

`ticket_flights.csv`. The `amount` column is the ticket/fare amount for a ticket on a flight.

**Which table contains booking date?**

`bookings.csv`, using `book_date`.

### Primary keys / important keys

| Table | Important key |
|---|---|
| aircrafts_data | `aircraft_code` |
| airports_data | `airport_code` |
| bookings | `book_ref` |
| flights | `flight_id` |
| seats | `aircraft_code + seat_no` |
| tickets | `ticket_no` |
| ticket_flights | `ticket_no + flight_id` |
| boarding_passes | `ticket_no + flight_id` (with boarding/seat details) |

### Important one-to-many relationships

One aircraft type has many seats and many flights. One flight has many boarding records and many ticket-flight records. One booking can contain multiple tickets. One ticket can be associated with one or more flight legs through `ticket_flights`.

# 2. First-Look / Data Quality — Answers

The source data contains 8 tables. The executed notebook reported these row counts:

| Table | Rows |
|---|---:|
| `aircrafts_data` | 9 |
| `airports_data` | 104 |
| `boarding_passes` | 579,686 |
| `bookings` | 262,788 |
| `flights` | 33,121 |
| `seats` | 1,339 |
| `ticket_flights` | 1,045,726 |
| `tickets` | 366,733 |

The first `flights.info()` output showed the date and status fields as `object`, so they needed conversion to datetime or suitable numeric/string types.

### Important missing-value lesson

The source represents missing values using `\N`. The initial `isnull().sum()` on the raw CSV-loaded data therefore showed zero missing values in some columns because `\N` was still being read as text. After replacing `\N` with `pd.NA`, those values become real missing observations.

For example, scheduled flights can have missing `actual_departure` and `actual_arrival`. That is meaningful information and should not simply be deleted.

**Inference:** The cleaning step is necessary before deciding how to handle missing data. **Missing is not automatically zero.**

# 3. JSON / Text Cleaning — Answer

Aircraft models and airport names/cities are stored in a JSON-like structure containing multiple languages. Extracting the English value makes the analysis readable.

Example: `773` becomes **Boeing 777-300** instead of a raw JSON string.

**Inference:** Cleaning improves interpretability. The analysis can be technically correct with the raw JSON, but business users cannot easily understand the result. Readable labels make charts and tables easier to communicate.

# 4. Aircraft Range — Detailed Answer

The uploaded SQL notebook reports:

| Code | Model | Range |
|---|---|---:|
| 773 | Boeing 777-300 | 11,100 |
| 763 | Boeing 767-300 | 7,900 |
| 319 | Airbus A319-100 | 6,700 |
| 320 | Airbus A320-200 | 5,700 |
| 321 | Airbus A321-200 | 5,600 |
| 733 | Boeing 737-300 | 4,200 |
| SU9 | Sukhoi Superjet-100 | 3,000 |
| CR2 | Bombardier CRJ-200 | 2,700 |
| CN1 | Cessna 208 Caravan | 1,200 |

### Prompt: Which aircraft has the greatest range?
**Answer:** Boeing 777-300 (`773`) with a range of 11,100.

### Prompt: Does range tell us whether occupancy is high?
**Answer:** No. Range describes how far an aircraft can operate; occupancy describes how much of its available seating capacity is used. They answer different business questions. The fleet contains both long-range and short-range aircraft, so aircraft purpose and route should be considered when evaluating performance.

**Inference:** Aircraft performance should be compared in context rather than assuming that the aircraft with the greatest range should also have the greatest occupancy.

# 5. Aircraft Capacity — Detailed Answer

The seat table contains the following capacities:

| Aircraft | Seats |
|---|---:|
| 773 | 402 |
| 763 | 222 |
| 321 | 170 |
| 733 | 130 |
| 320 | 140 |
| 319 | 116 |
| SU9 | 97 |
| CR2 | 50 |
| CN1 | 12 |

### Aircraft with more than 100 seats

`773`, `763`, `321`, `320`, `733`, and `319`.

### Largest-capacity aircraft

The **Boeing 777-300 (`773`)** has the largest capacity at **402 seats**.

### Why does large capacity create risk?

A larger aircraft gives the airline more seats to sell, so strong demand can create higher revenue potential. However, when demand is weak, the same aircraft can leave a larger absolute number of seats empty.

**Inference:** Capacity should always be interpreted alongside occupancy and route demand. A large aircraft is not automatically a problem or an advantage.

# 6. Seat Configuration by Fare Class — Detailed Answer

The original analysis found that cabin mix differs by aircraft. The main observations were:

- **A319** has the largest Business and Economy seat configurations in the original analysis.
- **Boeing 777-300 (`773`)** has the largest Comfort-class configuration.
- **CN1** and **CR2** have Economy-only configurations.

### Why does this matter?

Aircraft do not all sell the same product mix. An aircraft with substantial premium seating should not be evaluated only on overall occupancy; premium seats can contribute much higher revenue per passenger.

**Inference:** Pricing and capacity decisions should consider **aircraft + fare class**, not aircraft alone.

# 7. Ticket Booking Trend — Detailed Answer

The original executed analysis reported:

- ticket bookings increased gradually from **June 22 to July 7**;
- from **July 8 through August**, bookings were comparatively stable, with one noticeable peak;
- booking value followed a broadly similar pattern to ticket-booking volume.

### Why is this useful?

Time-based analysis helps the airline identify periods of stronger and weaker booking activity.

**Inference:** Booking activity changes over time, so the airline can monitor booking pace and investigate targeted commercial actions during weaker periods rather than applying the same strategy throughout the entire booking period.

### Important caution

A peak in one day should be investigated further. The EDA identifies the peak but does not explain its cause. Possible explanations require additional data such as promotions, route demand, holidays, events or pricing changes.

# 8. Fare Distribution — Detailed Answer

Ticket fares are spread across a wide range because the dataset contains multiple aircraft and fare classes. The fare-class analysis shows strong separation between Economy and premium classes.

Examples from the uploaded output:

- A319 Business average fare ≈ **113,551** versus Economy ≈ **38,311**.
- 763 Business ≈ **82,840** versus Economy ≈ **27,595**.
- 773 Business ≈ **57,780**, Comfort ≈ **32,741**, Economy ≈ **19,265**.
- SU9 Business ≈ **33,488** versus Economy ≈ **11,220**.

**Inference:** One overall average fare would hide important differences between fare classes and aircraft. Pricing analysis should therefore be segmented.

# 9. Aircraft Usage / Frequency — Detailed Answer

The executed pandas analysis reported:

| Aircraft | Flights |
|---|---:|
| CN1 | 9,273 |
| CR2 | 9,048 |
| SU9 | 8,504 |
| 321 | 1,952 |
| 733 | 1,274 |
| 319 | 1,239 |
| 763 | 1,221 |
| 773 | 610 |

### Most frequently used aircraft

**CN1** with **9,273 flights** in the executed flight-level data.

### Does the most-used aircraft also have the highest occupancy?

**No.** The original occupancy analysis reports CN1 at about **50.0%**, while 773 has the highest occupancy at about **65.9%**.

### Why analyse frequency + occupancy together?

An aircraft can have moderate occupancy but operate very frequently. Even a small improvement applied repeatedly can therefore affect a larger amount of seat capacity.

**Inference:** Commercial opportunities should consider both **performance** and **volume**.

# 10. Aircraft Occupancy — Detailed Answer

The original SQL notebook calculated average occupancy using flights with observed boarding records. The reported values were:

| Aircraft | Average Occupancy |
|---|---:|
| 773 | 65.9% |
| 733 | 61.7% |
| SU9 | 58.6% |
| 321 | 52.2% |
| 763 | 51.3% |
| CN1 | 50.0% |
| 319 | 46.2% |
| CR2 | 43.0% |

### Lowest occupancy
**CR2 — 43.0%**.

### Highest occupancy
**773 — 65.9%**.

### Inference
Occupancy differs materially across aircraft. This indicates that seat utilisation is not uniform. The business should investigate the routes, schedules, demand and fare mix associated with lower-occupancy aircraft before changing aircraft allocation.

### Critical correction to the earlier learner output
The old learner output showing a **0% median occupancy** was caused by converting missing boarding observations into zero passengers. That result should **not** be presented to students as the true occupancy distribution.

# 11. Fare Class vs Average Fare — Detailed Answer

The uploaded output shows a clear premium between Business and Economy in aircraft that offer both classes. The 773 also has a Comfort class between the two in the observed fare structure.

Some exact average fares from the output:

| Aircraft | Fare class | Avg fare |
|---|---|---:|
| 319 | Business | 113,550.558 |
| 319 | Economy | 38,311.402 |
| 321 | Business | 34,435.663 |
| 321 | Economy | 11,534.975 |
| 733 | Business | 41,865.626 |
| 733 | Economy | 13,985.152 |
| 763 | Business | 82,839.843 |
| 763 | Economy | 27,594.722 |
| 773 | Business | 57,779.909 |
| 773 | Comfort | 32,740.553 |
| 773 | Economy | 19,265.226 |
| CN1 | Economy | 6,568.552 |
| CR2 | Economy | 13,207.661 |
| SU9 | Business | 33,487.850 |
| SU9 | Economy | 11,220.183 |

### Inference
Premium fare classes generate higher revenue per passenger than Economy in the observed data. Therefore, the airline should not rely only on increasing passenger volume through blanket discounts; it should also protect premium yield where demand supports it.

# 12. Average Fare vs Occupancy — Detailed Answer

The uploaded executed analysis gives a Pearson correlation of approximately **-0.049** between average fare and occupancy.

This is a **very weak linear relationship**.

### Correct inference
> Average fare alone does not explain occupancy in this dataset. Route, aircraft type, fare class and timing should also be considered. The observed correlation is an association, not proof that changing price will cause occupancy to change.

### Business implication
Pricing experiments should be targeted and validated rather than assuming that lowering price will automatically create enough additional demand to justify the loss in yield.

**Teaching point:** Correlation ≠ causation.

# 13. Day of Week vs Occupancy — Detailed Answer

The earlier executed learner notebook reported these averages:

| Day | Occupancy |
|---|---:|
| Tuesday | 20.1% |
| Friday | 19.1% |
| Monday | 19.0% |
| Sunday | 18.2% |
| Saturday | 17.8% |
| Thursday | 15.8% |
| Wednesday | 15.3% |

### Important qualification
These exact percentages came from the earlier version that filled missing boarding observations with zero. Therefore they should be treated as **reference-only**, not final corrected values.

### Correct way to teach the inference
> Occupancy varies by day of week in the observed analysis. Wednesday and Thursday were lower than the other days in the previous output, while Tuesday was higher. However, the corrected notebook should be rerun using only flights with an observed boarding record before treating the exact percentages as final.

**Possible action:** Investigate booking pace, route mix and demand on weaker days. Do not assume that day itself causes lower demand.

# 14. Aircraft Revenue — Detailed Answer

From the aircraft/fare-class revenue output, total ticket revenue by aircraft is:

| Aircraft | Total ticket revenue | Avg revenue / ticket |
|---|---:|---:|
| SU9 | 5,114,484,700 | 29,056.93 |
| 763 | 4,371,277,100 | 55,316.52 |
| 773 | 3,431,205,500 | 34,829.98 |
| 319 | 2,706,163,100 | 78,583.01 |
| CR2 | 1,982,760,500 | 30,527.96 |
| 321 | 1,638,164,100 | 24,217.08 |
| 733 | 1,426,552,100 | 27,137.27 |
| CN1 | 96,373,800 | 11,705.79 |

### Highest total revenue
**SU9** with approximately **5.114 billion** in ticket revenue during the dataset period.

### Does highest revenue mean highest occupancy?
No. SU9 has approximately **58.6%** occupancy, while 773 has the highest occupancy at **65.9%**.

### Why not call one aircraft “best”?
Because different metrics answer different questions. SU9 has the highest total revenue, 773 has the highest occupancy, and A319 has the highest average revenue per ticket among the aircraft-level totals above. Each reflects a different dimension of performance.

### Corrected inference
> SU9 generates the highest total ticket revenue in the observed dataset. This result should be interpreted alongside ticket volume, average fare, aircraft capacity, flight frequency and occupancy rather than attributing the difference to a single cause.

# 15. Occupancy vs Revenue — Detailed Answer

At a general level, more occupied seats create more opportunities to earn ticket revenue, but total revenue also depends on:

- average fare
- fare mix
- aircraft capacity
- flight frequency
- route mix

Therefore, occupancy should not be used alone to judge commercial performance.

A useful metric is **revenue per available seat** because it combines revenue with the amount of capacity offered.

**Inference:** A route can have high occupancy but moderate yield, while another route can have lower occupancy but stronger revenue per available seat. This is why the business should monitor both utilisation and yield.

# 16. Route + Aircraft + Occupancy — Detailed Answer

The goal is to find repeated route-aircraft combinations with low occupancy. This is more informative than looking only at route or only at aircraft.

### Correct inference
> A route-aircraft combination with low occupancy, repeated operations and many empty seats is a stronger candidate for investigation than a route with low occupancy on only one flight.

### What to investigate
- fare and pricing
- booking pace
- day/time of operation
- route demand
- aircraft capacity
- passenger mix

### Important caution
The earlier executed learner output contained many rows with zero passengers because missing boarding records had been filled with zero. Those rows must **not** be called genuine zero-occupancy opportunities. The corrected final notebook must be rerun before publishing exact route names as answers.

# 17. Route + Aircraft + Revenue per Available Seat — Detailed Answer

The earlier executed route summary showed examples such as:

| Route | Aircraft | Occupancy | Revenue / available seat |
|---|---|---:|---:|
| DME → KHV | 763 | 41.3% | 55,640.105 |
| KHV → DME | 763 | 38.0% | 54,186.811 |
| LED → IKT | 321 | 42.0% | 39,691.707 |
| IKT → LED | 321 | 31.5% | 38,963.385 |
| KHV → LED | 763 | 23.0% | 37,488.731 |
| LED → KHV | 763 | 26.8% | 36,829.914 |

These values show why occupancy and revenue-per-available-seat can tell different stories. For example, DME → KHV has only about 41.3% occupancy but very high revenue per available seat in the old output.

### Qualification
These route values came from the earlier execution that included the boarding-missingness issue, so use them as **illustrative teaching examples**, not as final corrected route rankings. The final corrected notebook should be rerun before publishing exact route targets.

# 18. Low-Occupancy Opportunity Rule — Answer

The learner notebook uses a simple screening rule:

1. at least **10 observed flights**
2. occupancy below the **median occupancy** in the route dataset

This is a screening rule, not a final business decision.

### Correct inference
> These route-aircraft combinations have repeated observations and relatively low occupancy. They are candidates for further investigation, not automatically bad or unprofitable routes.

### Why use a minimum number of flights?
A route with one or two observations can produce an unstable result. Requiring repeated observations gives the analyst more evidence before recommending action.

# 19. 10% Occupancy Improvement Scenario — Detailed Answer

The original SQL analysis uses a **10% relative increase**, not a 10 percentage-point increase.

For example:

`46.2% × 1.10 = 50.8%`

The aircraft-level scenario is:

| Aircraft | Current occupancy | 10% relative uplift | Current revenue | Additional revenue (scenario) |
|---|---:|---:|---:|---:|
| 319 | 46.2% | 50.8% | 2,706,163,100 | 270,616,310 |
| 321 | 52.2% | 57.5% | 1,638,164,100 | 163,816,410 |
| 733 | 61.7% | 67.9% | 1,426,552,100 | 142,655,210 |
| 763 | 51.3% | 56.5% | 4,371,277,100 | 437,127,710 |
| 773 | 65.9% | 72.5% | 3,431,205,500 | 343,120,550 |
| CN1 | 50.0% | 55.0% | 96,373,800 | 9,637,380 |
| CR2 | 43.0% | 47.3% | 1,982,760,500 | 198,276,050 |
| SU9 | 58.6% | 64.4% | 5,114,484,700 | 511,448,470 |

### Largest additional revenue sensitivity
**SU9**, because it has the largest current revenue base.

### Total dataset-period sensitivity
Current ticket revenue ≈ **20,766,980,900**. A 10% proportional uplift gives approximately **2,076,698,090** additional revenue, producing a scenario revenue of about **22,843,678,990**.

### Critical interpretation
This is a **revenue sensitivity**, not a profit forecast. It assumes average yield remains unchanged and that the additional seats can actually be sold. The dataset does not contain fuel, labour, tax, maintenance, leasing or financing costs. Therefore, the extra revenue cannot be called extra profit.

Also, the original SQL notebook column called `inc Total Annual Turnover` is actually the **estimated revenue after the uplift**, not the incremental revenue. Incremental revenue is the difference between scenario revenue and current revenue.

# 20. Answers to the 10 Final Business Questions

### Question 1 — Which aircraft types have the highest and lowest occupancy?
**Highest:** 773 at about **65.9%**.  
**Lowest:** CR2 at about **43.0%**.

**Inference:** Occupancy differs materially across aircraft, so utilisation should be investigated by aircraft and route rather than using one fleet-wide assumption.

### Question 2 — Which aircraft types have the greatest available capacity?
**773** has the largest capacity at **402 seats**, followed by **763** at **222 seats** and **321** at **170 seats**.

**Inference:** Large-capacity aircraft create greater absolute empty-seat exposure when demand is weak.

### Question 3 — Does the most frequently used aircraft also have the highest occupancy?
No. **CN1** is the most frequently used in the executed flight table with **9,273 flights**, but **773** has the highest occupancy at about **65.9%**.

**Inference:** Frequency and utilisation need to be analysed together.

### Question 4 — How does fare class affect average fare?
Business generally has a higher average fare than Economy where both are available. The 773 also has a Comfort class. Aircraft differ in their fare-class mix.

**Inference:** Pricing should be segmented by fare class and aircraft rather than based on one average fare.

### Question 5 — Is average fare strongly related to occupancy?
No. The observed correlation is about **-0.049**, indicating a very weak linear relationship.

**Inference:** Average fare alone does not explain occupancy. Other variables and controlled pricing tests are needed.

### Question 6 — Does occupancy vary by day of week?
Yes, the earlier execution showed variation across days. The previous numbers should not be treated as final corrected estimates because that version converted missing boarding observations to zero.

**Inference:** Day-level differences are worth investigating, but the corrected occupancy population should be used before making operational decisions.

### Question 7 — Which aircraft generates the most ticket revenue?
**SU9**, with about **5.114 billion** in ticket revenue for the dataset period.

**Inference:** SU9 has the largest observed revenue base, but revenue alone does not establish that it is the strongest aircraft on every dimension.

### Question 8 — Which route-aircraft combinations appear to have the greatest unused capacity?
The corrected final answer must come from the final `route_analysis` output using only observed boarding records. The older output is contaminated by missing-boarding rows converted to zero.

**Correct inference:** prioritise repeated route-aircraft combinations with low occupancy and large empty-seat capacity.

### Question 9 — Which route-aircraft combinations should the business investigate first?
Use the simple rule: **at least 10 observed flights + occupancy below the route median**. Then sort by empty seats.

**Inference:** This produces a shortlist for investigation, not a definitive list of bad routes.

### Question 10 — What does the 10% occupancy scenario suggest?
A 10% relative occupancy improvement produces a proportional revenue sensitivity if average yield stays constant. SU9 has the largest absolute additional-revenue sensitivity because it already has the largest revenue base.

**Inference:** Occupancy improvement can have material revenue potential, but the scenario does not prove that the uplift is achievable or profitable.

# 21. Final Recommendation — Model Answer

> The analysis shows that occupancy varies substantially across aircraft, with CR2 at about 43.0% and 773 at about 65.9%. The airline should therefore avoid a single fleet-wide strategy. Aircraft frequency and capacity should be considered together with occupancy because frequently operated aircraft can create substantial cumulative unused capacity. Fare analysis shows strong differences between Economy and premium classes, so blanket discounting may weaken yield. Route-aircraft combinations with repeated low occupancy should be investigated using booking pace, route demand and aircraft allocation. The airline can test targeted pricing or schedule interventions on selected weak-demand segments while monitoring occupancy and revenue per available seat. However, the 10% uplift is only a revenue sensitivity and not a profit forecast because operating-cost data are not available.

# 22. What Students Should NOT Write

Avoid:

- “CR2 is the worst aircraft.”
- “773 is the best aircraft.”
- “Lower fares cause higher occupancy.”
- “SU9 has the highest revenue because its fares are low.”
- “These zero-passenger routes are definitely bad routes.”
- “A 10% occupancy increase will create 2.08 billion profit.”

Prefer:

- “CR2 has the lowest observed occupancy.”
- “773 has the highest observed occupancy.”
- “Average fare shows a very weak relationship with occupancy.”
- “SU9 has the highest observed ticket revenue; the difference should be examined alongside volume, fare and aircraft usage.”
- “Low-occupancy route-aircraft combinations are candidates for investigation.”
- “The 10% scenario indicates potential additional revenue under a constant-yield assumption.”

# 23. Teaching Note — The Most Important EDA Lessons

1. **Understand the grain before merging.**
2. **Aggregate one-to-many tables before creating a one-row-per-flight dataset.**
3. **Do not treat missing observations as zero without evidence.**
4. **Every chart should answer a business question.**
5. **Every inference should explain why the result matters.**
6. **Correlation is association, not causation.**
7. **Revenue is not profit when cost data are missing.**
8. **Do not choose a “best” aircraft from one KPI.**
9. **Use volume + performance when identifying business opportunities.**
10. **Treat scenarios as scenarios, not guaranteed forecasts.**